# HapMap3 panel for the round-1 set (ancestry PCA input)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
AMBIGUOUS = {("A", "T"), ("T", "A"), ("C", "G"), ("G", "C")}

def verify_panel(rep, prefix, label, ambiguous_ok=False):
    """Re-check the written panel rather than trust the commands that made it."""
    maf, geno, hwe = cfg.param("qc_maf"), cfg.param("qc_geno"), cfg.param("qc_hwe")
    bim = pd.read_csv(prefix + ".bim", sep="\t", header=None, names=["chr", "id", "cm", "pos", "a1", "a2"])
    if not ambiguous_ok:
        n_amb = sum((a, b) in AMBIGUOUS for a, b in zip(bim["a1"], bim["a2"]))
        rep.check("no strand-ambiguous SNPs (A/T, C/G)", n_amb == 0, f"{n_amb} found")
    rep.check("biallelic SNPs only", bim["a1"].isin(list("ACGT")).all() and bim["a2"].isin(list("ACGT")).all())
    rep.check("no duplicate variant IDs", not bim["id"].duplicated().any())
    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run([cfg.tool("plink2"), "--bfile", prefix, "--freq", "--missing", "variant-only",
                        "--hardy", "--silent", "--out", f"{tmp}/v"], check=True)
        frq = pd.read_csv(f"{tmp}/v.afreq", sep="\t")
        vmiss = pd.read_csv(f"{tmp}/v.vmiss", sep="\t")
        hardy = pd.read_csv(f"{tmp}/v.hardy", sep="\t")
    m = np.minimum(frq["ALT_FREQS"], 1 - frq["ALT_FREQS"])
    rep.check(f"MAF >= {maf} within {label}", m.min() >= maf, f"min {m.min():.4f}")
    rep.check(f"variant missingness <= {geno}", vmiss["F_MISS"].max() <= geno, f"max {vmiss['F_MISS'].max():.4f}")
    rep.check(f"HWE p >= {hwe:g}", hardy["P"].min() >= hwe, f"min {hardy['P'].min():.2e}")

def report_counts(rep, prefix):
    counts = pd.read_csv(prefix + ".counts.tsv", sep="\t")
    rep.count("n_samples", counts.set_index("step").loc["samples", "variants"])
    steps = counts[counts["step"] != "samples"]
    rep.table(steps, "variants_by_step")
    rep.note(n_variants=int(steps["variants"].iloc[-1]))
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(steps["step"], steps["variants"]); ax.invert_yaxis(); ax.set_xlabel("variants")
    rep.fig(fig, "variants_by_step"); plt.show()

In [ ]:
STAGE = "hm3_panel"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
PREFIX = OUT[:-4]
RUN = needs_run(OUT, FORCE)
geno_key = "acaf_bfile" if "acaf_bfile" in cfg.params["inputs"] else "acaf_pfile"
setenv(PLINK2=cfg.tool("plink2"), PLINK=cfg.tool("plink"),
       GENO_FLAG="--bfile" if geno_key == "acaf_bfile" else "--pfile",
       GENO_PATTERN=cfg.input(geno_key, chr="{chr}"), NCHR=22, EXTRACT=cfg.input("hm3_ids"),
       MAF=cfg.param("qc_maf"), HWE=cfg.param("qc_hwe"), GENO=cfg.param("qc_geno"), THREADS=THREADS)
setenv(RUN=int(RUN), KEEP=single_target(cfg, "ancestry_round1"), REF_BFILE=cfg.input("ref_bfile"), OUT=PREFIX)

## Build

In [ ]:
%%bash
set -euo pipefail
[ "$RUN" = 1 ] || { echo "exists, keeping it (FORCE = True to remake)"; exit 0; }
mkdir -p "$(dirname "$OUT")"
work=$(mktemp -d); trap 'rm -rf "$work"' EXIT
n() { wc -l < "$1" | tr -d " "; }
printf 'step\tvariants\n' > "$work/counts.tsv"

# sample set, HapMap3, biallelic SNPs, per chromosome; IDs relabelled to chr:pos:ref:alt (no "chr")
: > "$work/merge.txt"; n_in=0
for c in $(seq 1 "$NCHR"); do
  "$PLINK2" "$GENO_FLAG" "${GENO_PATTERN//\{chr\}/$c}" --keep "$KEEP" \
    --set-all-var-ids '@:#:$r:$a' --new-id-max-allele-len 1000 --extract "$EXTRACT" \
    --snps-only just-acgt --max-alleles 2 --rm-dup exclude-all \
    --threads "$THREADS" --silent --make-pgen --out "$work/chr$c"
  n_in=$((n_in + $(grep -vc '^#' "$work/chr$c.pvar")))
  echo "$work/chr$c" >> "$work/merge.txt"
done
printf 'hm3_biallelic_snps\t%s\n' "$n_in" >> "$work/counts.tsv"
"$PLINK2" --pmerge-list "$work/merge.txt" pfile --threads "$THREADS" --silent --make-pgen --out "$work/merged"

# QC within this sample set
"$PLINK2" --pfile "$work/merged" --geno "$GENO" --maf "$MAF" --hwe "$HWE" \
  --threads "$THREADS" --silent --write-snplist --out "$work/qc"
printf 'after_qc\t%s\n' "$(n "$work/qc.snplist")" >> "$work/counts.tsv"

# drop A/T and C/G: their strand can't be checked against the reference
awk 'BEGIN{OFS="\t"} !/^#/ {r=$4; a=$5;
     if (!((r=="A"&&a=="T")||(r=="T"&&a=="A")||(r=="C"&&a=="G")||(r=="G"&&a=="C"))) print $3, $4, $5}' \
  "$work/merged.pvar" | LC_ALL=C sort > "$work/unamb.tsv"
cut -f1 "$work/unamb.tsv" | LC_ALL=C sort | LC_ALL=C comm -12 <(LC_ALL=C sort "$work/qc.snplist") - > "$work/keep.ids"
printf 'unambiguous\t%s\n' "$(n "$work/keep.ids")" >> "$work/counts.tsv"

# keep variants that match the reference on ID + REF + ALT
"$PLINK2" --bfile "$REF_BFILE" --set-all-var-ids '@:#:$r:$a' --new-id-max-allele-len 1000 \
  --extract "$work/keep.ids" --freq counts --threads "$THREADS" --silent --out "$work/ref"
awk 'NR>1 {print $2"\t"$3"\t"$4}' "$work/ref.acount" | LC_ALL=C sort > "$work/ref.tsv"
LC_ALL=C comm -12 "$work/unamb.tsv" "$work/ref.tsv" | cut -f1 | LC_ALL=C sort \
  | LC_ALL=C comm -12 "$work/keep.ids" - > "$work/final.ids"
printf 'matched_to_reference\t%s\n' "$(n "$work/final.ids")" >> "$work/counts.tsv"

"$PLINK2" --pfile "$work/merged" --extract "$work/final.ids" --threads "$THREADS" --silent --make-bed --out "$OUT"
printf 'final\t%s\nsamples\t%s\n' "$(n "$OUT.bim")" "$(n "$OUT.fam")" >> "$work/counts.tsv"
cp "$work/counts.tsv" "$OUT.counts.tsv"; column -t "$OUT.counts.tsv"

In [ ]:
if RUN:
    mark_done(STAGE, OUT, tools={"plink2": cfg.tool("plink2")})

## Checks

In [ ]:
report_counts(rep, PREFIX)
verify_panel(rep, PREFIX, "the round-1 set")
rep.save()